In [1]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import load_model

# === Step 1: User Inputs ===
symbol = input("Enter stock symbol (e.g., AAPL): ").upper()
time_steps = int(input("Enter number of lookback days (e.g., 60): "))

# === Step 2: Load Model ===
model_path = f"models/{symbol}_best_model.h5"
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model file not found: {model_path}")

model = load_model(model_path)

# === Step 3: Load Stock Data ===
stock_path = f"../LSTM/datasets/{symbol}_daily_data.csv"
if not os.path.exists(stock_path):
    raise FileNotFoundError(f"Stock data file not found: {stock_path}")

stock_df = pd.read_csv(stock_path)
stock_df = stock_df.iloc[1:].reset_index(drop=True)
stock_df['Date'] = pd.to_datetime(stock_df['Date'])

# === Step 4: Load Sentiment Data ===
today = datetime.today().strftime('%Y-%m-%d')
sentiment_path = f"../Sentiment-Analysis/sentiment/{today}/{symbol}_sentiment.csv"
if not os.path.exists(sentiment_path):
    raise FileNotFoundError(f"Sentiment data file not found: {sentiment_path}")

sentiment_df = pd.read_csv(sentiment_path)
sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])
sentiment_df['sentiment_score'] = sentiment_df['sentiment'].map({
    'POSITIVE': 1, 'NEGATIVE': -1, 'NEUTRAL': 0
})

# === Step 5: Merge and Preprocess Data ===
daily_sentiment = sentiment_df.groupby('date')['sentiment_score'].mean().reset_index()
daily_sentiment.columns = ['Date', 'Sentiment']

merged_df = pd.merge(stock_df[['Date', 'Close']], daily_sentiment, on='Date', how='left')
merged_df['Sentiment'].fillna(0, inplace=True)

# Ensure we have enough data
if len(merged_df) < time_steps:
    raise ValueError(f"Not enough data to generate {time_steps} time steps.")

# === Step 6: Scale ===
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(merged_df[['Close', 'Sentiment']])

# === Step 7: Get Last Sequence ===
last_sequence = scaled_data[-time_steps:]  # shape: (time_steps, 2)
last_sequence = np.expand_dims(last_sequence, axis=0)  # shape: (1, time_steps, 2)

# === Step 8: Predict ===
scaled_prediction = model.predict(last_sequence)
combined = np.concatenate([scaled_prediction, np.zeros_like(scaled_prediction)], axis=1)
predicted_price = scaler.inverse_transform(combined)[0][0]

# === Step 9: Output ===
print(f"\n📈 Predicted next closing price for {symbol}: ${predicted_price:.2f}")


2025-07-13 06:44:22.353367: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-13 06:44:22.356420: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-13 06:44:22.364929: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752389062.378843   65380 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752389062.383259   65380 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752389062.394700   65380 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step

📈 Predicted next closing price for TSLA: $303.72
